In [ ]:
# 读取Output目录下*ranked_by_number.csv文件，记录总共不包含标题的行数，并与/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/datasets/PepSet_3per_length/selected_pdbs.txt中的pdb一一对应上
#!/usr/bin/env python3
import os
import re
import pandas as pd
filter = 1
res = '020'
temperature = '0.1'


In [1]:
# 按照 overall_confidence = exp[-mean_over_residues(log_probs for native sequence)] 计算 overall_confidence
import torch
import os
import pandas as pd

def overall_confidence_from_score_pt(pt_path: str):
    d = torch.load(pt_path, map_location="cpu")
    # inputs from score.py
    log_probs = torch.tensor(d["log_probs"], dtype=torch.float32)  # [N, L, 21]
    native_seq = torch.tensor(d["native_sequence"], dtype=torch.long)  # [L]
    mask = torch.tensor(d["mask"], dtype=torch.float32)  # [L]
    chain_mask = torch.tensor(d["chain_mask"], dtype=torch.float32)  # [L]
    m = mask * chain_mask  # [L]

    # broadcast native sequence across all samples N
    N, L, _ = log_probs.shape
    S_one_hot = torch.nn.functional.one_hot(native_seq, num_classes=21).float()  # [L, 21]
    S_one_hot = S_one_hot.unsqueeze(0).repeat(N, 1, 1)  # [N, L, 21]

    loss_per_pos = -(S_one_hot * log_probs).sum(-1)  # [N, L]
    loss = (loss_per_pos * m).sum(-1) / (m.sum() + 1e-8)  # [N]
    overall_confidence = torch.exp(-loss)  # [N]
    return overall_confidence[0].item()

dc = []
for file in os.listdir('/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-0.2T/ligandmpnn_v_32_030_25/score/'):
    if file.endswith('.pt'):
        path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-0.2T/ligandmpnn_v_32_030_25/score/{file}'
        conf = overall_confidence_from_score_pt(path)
        dc.append((file, conf))
df = pd.DataFrame(dc, columns=['pt', 'overall_confidence'])
df

,pt,overall_confidence
0,5d94_2845.pt,0.243210
1,5d94_7098.pt,0.245408
2,5d94_367.pt,0.275168
3,5d94_8892.pt,0.212331
4,5d94_162.pt,0.264364
...,...,...
2780,5d94_5675.pt,0.232430
2781,5d94_90.pt,0.275194
2782,5d94_353.pt,0.254723
2783,5d94_7429.pt,0.254018


In [ ]:
import torch, os

def losses_from_score_pt(pt_path: str):
    d = torch.load(pt_path, map_location="cpu")
    log_probs = torch.tensor(d["log_probs"], dtype=torch.float32)   # [N, L, 21]
    native_seq = torch.tensor(d["native_sequence"], dtype=torch.long)  # [L]
    mask = torch.tensor(d["mask"], dtype=torch.float32)             # [L]
    chain_mask = torch.tensor(d["chain_mask"], dtype=torch.float32) # [L]
    m = mask * chain_mask                                           # [L]

    


data = torch.load('/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-0.1T/ligandmpnn_v_32_020_25/score/5d94_1.pt', map_location="cpu")
print(data['log_probs'][0][0]) # 第一个batch的log_probs
# print(data['log_probs'][0].sum(1)) # 第一个batch的log_probs按最后一维求和
print(data['decoding_order'][0][0])
# print(data['native_sequence'])
# print(data['decoding_order'][0])



[-3.0433197  -5.2311134  -3.9932947  -3.8886251  -4.1093907  -2.4314733
 -2.8419075  -5.132772   -2.6364655  -3.8319345  -3.3641882  -3.2147875
 -3.371737   -3.3912473  -0.86153877 -3.5440392  -4.20629    -4.9966354
 -5.055893   -3.8975186  -5.5838194 ]
[ 0.  8.  5.  7.  9. 10.  1.  2.  6.  4.  3. 11.]
[14  9  3 19  3  3 17 16  3  3  3  7]
